# Shoaib HAR Preprocessing

Prepare the Shoaib example dataset for ZARA. This notebook creates fixed windows for all placements and saves database/test splits.


In [1]:
import os
import pickle
import pandas as pd
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

def check_label_continuity(df):
    """Find contiguous activity segments for each subject in the raw Shoaib dataframe."""
    continuity_segments = {}

    for subject in df['subject'].unique():
        subject_data = df[df['subject'] == subject]
        assert subject_data.index[0]==0

        for label in subject_data['activity'].unique():
            label_data = subject_data[subject_data['activity'] == label]

            indices = label_data.index
            segments = []
            start_idx = indices[0]

            for i in range(len(indices) - 1):
                if indices[i] + 1 != indices[i + 1]:
                    end_idx = indices[i]
                    segments.append((start_idx, end_idx))
                    start_idx = indices[i + 1]

            segments.append((start_idx, indices[-1]))

            if segments:
                continuity_segments[(subject, label)] = segments

    return continuity_segments

def split_sequences(sequences, window_size, stride):
    """Split a continuous sensor sequence into fixed-size windows with stride."""
    assert len(sequences[0]) == 30
    has_null = any(any(pd.isnull(item) or item == '' for item in sublist) for sublist in sequences) # Report missing values before failing.
    if has_null:
        raise ValueError("Has null values")

    segments = []
    labels = []

    num_complete_segments = (len(sequences) - window_size) // stride + 1

    for i in range(num_complete_segments):
        start = i * stride
        end = start + window_size
        segment = sequences[start:end]
        assert len(segment) == window_size
        segments.append(np.array(segment))
        labels.append([start, end-1])

    if labels[-1][1] < len(sequences) - 1:
        start = len(sequences) - window_size
        end = len(sequences)
        segment = sequences[start:end]
        assert len(segment) == window_size
        segments.append(np.array(segment))
        labels.append([start, end-1])
    assert len(labels) == len(segments)
    print(f"sequence length: {len(sequences)}\nsegments: {len(segments)}")
    pd.set_option('display.max_columns', None)
    print(labels[:5])
    print(labels[-5:])
    return segments, labels

activity_map = {
    0: "walking",
    1: "standing",
    2: "jogging",
    3: "sitting",
    4: "biking",
    5: "upstairs",
    6: "downstairs",
}

label2id = {}
id2label = {}
for i, (key, value) in enumerate(activity_map.items()):
    label2id[value]=i
    id2label[i]=value
print(f"label2id:\n{label2id}")
print(f"id2label:\n{id2label}")


label2id:
{'walking': 0, 'standing': 1, 'jogging': 2, 'sitting': 3, 'biking': 4, 'upstairs': 5, 'downstairs': 6}
id2label:
{0: 'walking', 1: 'standing', 2: 'jogging', 3: 'sitting', 4: 'biking', 5: 'upstairs', 6: 'downstairs'}


## Build Dataset Splits

Load participant CSV files, normalize column names, split continuous activity blocks into windows, and assign database/test splits.


In [2]:
test_id = ['subject1', 'subject9']
window_size_database=100
window_size_test=100
stride_database=100
stride_test=100

all_database_segments = []
all_database_labels = []

left_pocket_database_segments = []
right_pocket_database_segments = []
wrist_database_segments = []
upper_arm_database_segments = []
belt_database_segments = []

all_test_segments = []
all_test_labels = []

for i in range(1, 11):
    # Read the CSV and flatten the multi-level column names.
    df = pd.read_csv(f'./ori_data/Shoaib/Participant_{i}.csv', header=[0, 1])
    df.columns = df.columns.map(lambda x: f"{x[0]}_{x[1]}" if isinstance(x, tuple) else x)

    # Select the sensor columns by position.
    selected_indices = [1, 2, 3, 7, 8, 9, 15, 16, 17, 21, 22, 23, 29, 30, 31,
                        35, 36, 37, 43, 44, 45, 49, 50, 51, 57, 58, 59, 63, 64, 65, 69]

    df = df.iloc[:, selected_indices]

    # Capture the selected column names.
    selected_colnames = df.columns.tolist()

    # Map raw column names to normalized sensor names.
    rename_map = {
        selected_colnames[0]: 'acc_left_pocket_x',
        selected_colnames[1]: 'acc_left_pocket_y',
        selected_colnames[2]: 'acc_left_pocket_z',
        selected_colnames[3]: 'gyr_left_pocket_x',
        selected_colnames[4]: 'gyr_left_pocket_y',
        selected_colnames[5]: 'gyr_left_pocket_z',
        selected_colnames[6]: 'acc_right_pocket_x',
        selected_colnames[7]: 'acc_right_pocket_y',
        selected_colnames[8]: 'acc_right_pocket_z',
        selected_colnames[9]: 'gyr_right_pocket_x',
        selected_colnames[10]: 'gyr_right_pocket_y',
        selected_colnames[11]: 'gyr_right_pocket_z',
        selected_colnames[12]: 'acc_wrist_x',
        selected_colnames[13]: 'acc_wrist_y',
        selected_colnames[14]: 'acc_wrist_z',
        selected_colnames[15]: 'gyr_wrist_x',
        selected_colnames[16]: 'gyr_wrist_y',
        selected_colnames[17]: 'gyr_wrist_z',
        selected_colnames[18]: 'acc_upper_arm_x',
        selected_colnames[19]: 'acc_upper_arm_y',
        selected_colnames[20]: 'acc_upper_arm_z',
        selected_colnames[21]: 'gyr_upper_arm_x',
        selected_colnames[22]: 'gyr_upper_arm_y',
        selected_colnames[23]: 'gyr_upper_arm_z',
        selected_colnames[24]: 'acc_belt_x',
        selected_colnames[25]: 'acc_belt_y',
        selected_colnames[26]: 'acc_belt_z',
        selected_colnames[27]: 'gyr_belt_x',
        selected_colnames[28]: 'gyr_belt_y',
        selected_colnames[29]: 'gyr_belt_z',
        selected_colnames[30]: 'activity',
    }

    # Apply normalized column names.
    df = df.rename(columns=rename_map)

    df['activity'] = df['activity'].replace('running', 'jogging')
    df['activity'] = df['activity'].replace('upsatirs', 'upstairs')
    df['subject'] = f'subject{i}'
    continuity_segments = check_label_continuity(df)
    for key, value in continuity_segments.items():
        print(f"Subject: {key[0]} Activity: {key[1]}")
        for segment in value:
            # Split the continuous sequence into fixed windows.
            rows = df.loc[segment[0]:segment[1]]

            assert len(rows['subject'].unique()) == 1
            assert rows['subject'].unique()[0] == key[0]
            assert len(rows['activity'].unique()) == 1, f"Subject {key[0]}, activity {key[1]} but has {rows['activity'].unique()},  {segment[0]} to {segment[1]}"
            assert rows['activity'].unique()[0] == key[1]

            subject_activity_df = rows.iloc[:, ~rows.columns.isin(['subject', 'activity'])]
            subject_activity_series = subject_activity_df.values.tolist()

            if key[0] not in test_id:
                segments, labels = split_sequences(subject_activity_series, window_size_database, stride_database)
                segments = np.array(segments)
                all_database_segments.extend(segments)
                left_pocket_database_segments.extend(segments[:, :, :6])
                right_pocket_database_segments.extend(segments[:, :, 6:12])
                wrist_database_segments.extend(segments[:, :, 12:18])
                upper_arm_database_segments.extend(segments[:, :, 18:24])
                belt_database_segments.extend(segments[:, :, 24:])
            else:
                segments, labels = split_sequences(subject_activity_series, window_size_test, stride_test)
                segments = np.array(segments)
                all_test_segments.extend(segments)

            for label in labels:
                label_dict = {
                    "subject": key[0],
                    "activity_name": key[1],
                    "activity":  label2id[key[1]],
                    "segments": label
                }
                if key[0] not in test_id:
                    all_database_labels.append(label_dict)
                else:
                    all_test_labels.append(label_dict)
        print("------"*10)

print(f"all_database_segments: {len(all_database_segments)}")
print(f"left_pocket_database_segments: {len(left_pocket_database_segments)}")
print(f"right_pocket_database_segments: {len(right_pocket_database_segments)}")
print(f"wrist_database_segments: {len(wrist_database_segments)}")
print(f"upper_arm_database_segments: {len(upper_arm_database_segments)}")
print(f"belt_database_segments: {len(belt_database_segments)}")
print(f"all_database_labels: {len(all_database_labels)}")
print(f"all_test_segments: {len(all_test_segments)}")
print(f"all_test_labels: {len(all_test_labels)}")


Subject: subject1 Activity: walking
sequence length: 9000
segments: 90
[[0, 99], [100, 199], [200, 299], [300, 399], [400, 499]]
[[8500, 8599], [8600, 8699], [8700, 8799], [8800, 8899], [8900, 8999]]
------------------------------------------------------------
Subject: subject1 Activity: standing
sequence length: 9000
segments: 90
[[0, 99], [100, 199], [200, 299], [300, 399], [400, 499]]
[[8500, 8599], [8600, 8699], [8700, 8799], [8800, 8899], [8900, 8999]]
------------------------------------------------------------
Subject: subject1 Activity: jogging
sequence length: 9000
segments: 90
[[0, 99], [100, 199], [200, 299], [300, 399], [400, 499]]
[[8500, 8599], [8600, 8699], [8700, 8799], [8800, 8899], [8900, 8999]]
------------------------------------------------------------
Subject: subject1 Activity: sitting
sequence length: 9000
segments: 90
[[0, 99], [100, 199], [200, 299], [300, 399], [400, 499]]
[[8500, 8599], [8600, 8699], [8700, 8799], [8800, 8899], [8900, 8999]]
----------------

## Test Label Distribution

Inspect the raw held-out test split activity distribution before balanced sampling.


In [3]:
from collections import Counter

# Extract activity labels.
test_activity_labels = [label['activity_name'] for label in all_test_labels]

# Count samples per activity.
activity_distribution = Counter(test_activity_labels)

# Print sample counts per activity.
for activity_id, count in activity_distribution.items():
    print(f"{activity_id}: {count}")


walking: 180
standing: 180
jogging: 180
sitting: 180
biking: 180
upstairs: 180
downstairs: 180


## Balanced Test Subset

Select a subject-aware balanced subset of test examples for reproducible inference evaluation.


In [4]:
import random
from collections import defaultdict

def split_balanced_by_activity_subject(all_labels, N, seed=42):
    """
    all_labels: list of dicts, each with keys "activity" and "subject"
    N: total number of samples to select per activity
    Returns: list of selected indices (length ≈ N * #activities)
    """
    random.seed(seed)

    # Group indices by activity → subject → [indices]
    by_act = defaultdict(lambda: defaultdict(list))
    for i, lbl in enumerate(all_labels):
        by_act[lbl["activity"]][lbl["subject"]].append(i)

    selected_indices = []

    for activity, subj2inds in by_act.items():
        # Flatten into (subject, [indices]) list
        subject_pools = [(subj, inds[:]) for subj, inds in subj2inds.items()]
        random.shuffle(subject_pools)

        # Shuffle within each subject
        for _, inds in subject_pools:
            random.shuffle(inds)

        total_collected = 0
        temp_selected = []

        # 1st pass: try to fairly distribute quota
        S = len(subject_pools)
        base, rem = divmod(N, S)

        # Try base + (1 if rem > 0) per subject
        for i, (subj, inds) in enumerate(subject_pools):
            quota = base + (1 if i < rem else 0)
            taken = inds[:quota]
            temp_selected.extend(taken)
            total_collected += len(taken)
            # update leftover
            subject_pools[i] = (subj, inds[quota:])

        # 2nd pass: top up remaining from any subject with leftovers
        if total_collected < N:
            remaining = N - total_collected
            flat_leftover = [idx for _, inds in subject_pools for idx in inds]
            random.shuffle(flat_leftover)
            temp_selected.extend(flat_leftover[:remaining])

        selected_indices.extend(temp_selected)

    return selected_indices

# usage:
set_idx = split_balanced_by_activity_subject(
    all_test_labels,
    N=30,    # e.g. 25 per activity in split
    seed=SEED
)

all_test_segments = [all_test_segments[i] for i in set_idx]
all_test_labels   = [all_test_labels[i]   for i in set_idx]

from collections import defaultdict, Counter

# Count activity samples by subject.
activity_subject_counts1 = defaultdict(Counter)
for lbl in all_test_labels:
    act = lbl["activity"]
    subj = lbl["subject"]
    activity_subject_counts1[act][subj] += 1

# Print the subject distribution for each activity.
for act, subj_counts in activity_subject_counts1.items():
    print(f"Activity = {act}")
    for subj, cnt in subj_counts.items():
        print(f"    Subject {subj}: {cnt}")
    print()


Activity = 0
    Subject subject9: 15
    Subject subject1: 15

Activity = 1
    Subject subject9: 15
    Subject subject1: 15

Activity = 2
    Subject subject1: 15
    Subject subject9: 15

Activity = 3
    Subject subject1: 15
    Subject subject9: 15

Activity = 4
    Subject subject9: 15
    Subject subject1: 15

Activity = 5
    Subject subject9: 15
    Subject subject1: 15

Activity = 6
    Subject subject9: 15
    Subject subject1: 15



## Save Combined Dataset

Persist combined Shoaib database and test windows with labels.


In [5]:
output_path = "./dataset/shoaib"

with open(os.path.join(output_path, 'shoaib_database_segments.pkl'), 'wb') as f:
    pickle.dump(all_database_segments, f)

with open(os.path.join(output_path, 'shoaib_test_data.pkl'), 'wb') as f:
    pickle.dump(all_test_segments, f)

with open(os.path.join(output_path, 'shoaib_database_labels.pkl'), 'wb') as f:
    pickle.dump(all_database_labels, f)

with open(os.path.join(output_path, 'shoaib_test_labels.pkl'), 'wb') as f:
    pickle.dump(all_test_labels, f)


## Save Placement-Specific Dataset

Persist placement-specific database windows used by the RRF retrieval example.


In [6]:
with open(os.path.join(output_path, 'shoaib_left_pocket_database_segments.pkl'), 'wb') as f:
    pickle.dump(left_pocket_database_segments, f)

with open(os.path.join(output_path, 'shoaib_right_pocket_database_segments.pkl'), 'wb') as f:
    pickle.dump(right_pocket_database_segments, f)

with open(os.path.join(output_path, 'shoaib_wrist_database_segments.pkl'), 'wb') as f:
    pickle.dump(wrist_database_segments, f)

with open(os.path.join(output_path, 'shoaib_upper_arm_database_segments.pkl'), 'wb') as f:
    pickle.dump(upper_arm_database_segments, f)

with open(os.path.join(output_path, 'shoaib_belt_database_segments.pkl'), 'wb') as f:
    pickle.dump(belt_database_segments, f)
